# Exercise 08b — Temporal features and spatial residual CNNs

The TaxiBJ21 arrays and split code are provided. Tune temporal inputs and CNN capacity on validation data, freeze one setting per model family, and inspect the final test errors in space and time.

## Open this notebook

- **Colab.** Open the notebook, then run the setup cell below before any data-loading cell. It clones the public course repository and changes into `08 Convolutional neural networks`, so relative data paths work.
- **On your own computer.** Open the notebook from its lesson folder so the data files sit beside it.


In [ ]:
from pathlib import Path
import os

def in_colab():
    try:
        import google.colab  # noqa: F401
        return True
    except ImportError:
        return False

if in_colab():
    print("Running in Google Colab")
    REPO_URL = "https://github.com/hku-kejintao/AI-Code.git"
    LESSON_DIR = "08 Convolutional neural networks"
    base = Path("/content")
    repo = base / "AI-Code"
    os.chdir(base)
    if not repo.exists():
        !git clone {REPO_URL}
    lesson_path = repo / LESSON_DIR
    if not lesson_path.exists():
        raise FileNotFoundError(f"Lesson folder not found: {lesson_path}")
    os.chdir(lesson_path)
else:
    print("Running on your own computer")

print("Current folder:", os.getcwd())


### Check the packages

The cell should print version numbers and the word `ready`. NumPy holds arrays, pandas holds tables, Matplotlib draws figures, and scikit-learn provides preprocessing, models, and metrics. PyTorch builds the neural networks used in this lesson.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
import torch

print("numpy", np.__version__)
print("pandas", pd.__version__)
print("matplotlib", plt.matplotlib.__version__)
print("scikit-learn", sklearn.__version__)
print("torch", torch.__version__)
print("ready")


### Install, only if the check failed

Skip this cell when the check printed `ready`. After installation finishes, restart the kernel and run the cells above again.


In [ ]:
%pip install numpy pandas matplotlib scikit-learn torch


## Prepared data and reusable models

Each sample contains lagged inflow/outflow maps. A Ridge model supplies a stable per-cell temporal forecast; the CNN learns only a spatial correction. Initialising the correction to zero guarantees that training starts from the Ridge forecast rather than an arbitrary image.

In [ ]:
import copy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.linear_model import LinearRegression, Ridge

DATA_FILE = "taxibj21_small.npz"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(42)
np.random.seed(42)
torch.set_num_threads(2)

flow = np.load(DATA_FILE)["flow"]
print("flow:", flow.shape, "= (time, inflow/outflow, grid row, grid column)")
print("stored range:", (float(flow.min()), float(flow.max())), "relative intensity")

def make_flow_examples(flow_array, lags):
    inputs, targets, target_times = [], [], []
    for target_time in range(max(lags), len(flow_array)):
        lagged_maps = [flow_array[target_time - lag] for lag in lags]
        inputs.append(np.concatenate(lagged_maps, axis=0))
        targets.append(flow_array[target_time])
        target_times.append(target_time)
    return (
        np.asarray(inputs, dtype="float32"),
        np.asarray(targets, dtype="float32"),
        np.asarray(target_times),
    )

def grid_arrays(X, y):
    X_grid = X.transpose(0, 2, 3, 1).reshape(-1, X.shape[1])
    y_grid = y.transpose(0, 2, 3, 1).reshape(-1, 2)
    return X_grid, y_grid

def grid_to_maps(prediction, n_samples):
    return prediction.reshape(n_samples, 32, 32, 2).transpose(0, 3, 1, 2)

def flow_scores(name, prediction, target):
    error = prediction - target
    return {"model": name, "MAE": np.abs(error).mean(), "RMSE": np.sqrt((error ** 2).mean())}

class SpatialResidualCNN(nn.Module):
    def __init__(self, in_channels, hidden=32):
        super().__init__()
        self.temporal_projection = nn.Conv2d(in_channels, 2, 1)
        self.spatial_correction = nn.Sequential(
            nn.Conv2d(in_channels, hidden, 3, padding=1), nn.ReLU(),
            nn.Conv2d(hidden, hidden, 3, padding=1), nn.ReLU(),
            nn.Conv2d(hidden, 2, 3, padding=1),
        )
        nn.init.zeros_(self.spatial_correction[-1].weight)
        nn.init.zeros_(self.spatial_correction[-1].bias)

    def forward(self, x):
        return self.temporal_projection(x) + self.spatial_correction(x)

class MultiBranchResidualCNN(nn.Module):
    def __init__(self, hidden=12):
        super().__init__()
        self.temporal_projection = nn.Conv2d(12, 2, 1)
        self.recent_branch = nn.Sequential(nn.Conv2d(8, hidden, 3, padding=1), nn.ReLU())
        self.daily_branch = nn.Sequential(nn.Conv2d(2, hidden, 3, padding=1), nn.ReLU())
        self.weekly_branch = nn.Sequential(nn.Conv2d(2, hidden, 3, padding=1), nn.ReLU())
        self.fusion = nn.Sequential(
            nn.Conv2d(3 * hidden, hidden, 3, padding=1), nn.ReLU(),
            nn.Conv2d(hidden, 2, 3, padding=1),
        )
        nn.init.zeros_(self.fusion[-1].weight)
        nn.init.zeros_(self.fusion[-1].bias)

    def forward(self, x):
        recent = self.recent_branch(x[:, :8])
        daily = self.daily_branch(x[:, 8:10])
        weekly = self.weekly_branch(x[:, 10:12])
        correction = self.fusion(torch.cat([recent, daily, weekly], dim=1))
        return self.temporal_projection(x) + correction

def initialise_temporal_projection(model, ridge_model):
    with torch.no_grad():
        weights = torch.tensor(ridge_model.coef_, dtype=torch.float32)
        bias = torch.tensor(ridge_model.intercept_, dtype=torch.float32)
        model.temporal_projection.weight.copy_(weights[:, :, None, None])
        model.temporal_projection.bias.copy_(bias)
    for parameter in model.temporal_projection.parameters():
        parameter.requires_grad = False

def fit_flow(model, X_train, y_train, X_valid, y_valid, epochs=12):
    train_loader = DataLoader(TensorDataset(torch.tensor(X_train), torch.tensor(y_train)), 64, shuffle=True)
    valid_loader = DataLoader(TensorDataset(torch.tensor(X_valid), torch.tensor(y_valid)), 64)
    optimizer = torch.optim.Adam((p for p in model.parameters() if p.requires_grad), lr=1e-3)
    loss_function = nn.MSELoss()
    best_loss = float("inf")
    best_state = copy.deepcopy(model.state_dict())
    history = []
    for epoch in range(epochs):
        model.train()
        for xb, yb in train_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            optimizer.zero_grad()
            loss = loss_function(model(xb), yb)
            loss.backward()
            optimizer.step()
        model.eval()
        total = 0
        with torch.no_grad():
            for xb, yb in valid_loader:
                total += loss_function(model(xb.to(DEVICE)), yb.to(DEVICE)).item() * len(xb)
        valid_loss = total / len(valid_loader.dataset)
        history.append(valid_loss)
        if valid_loss < best_loss:
            best_loss = valid_loss
            best_state = copy.deepcopy(model.state_dict())
    model.load_state_dict(best_state)
    return history

def predict_flow(model, X):
    model.eval()
    outputs = []
    with torch.no_grad():
        for (xb,) in DataLoader(TensorDataset(torch.tensor(X)), 64):
            outputs.append(model(xb.to(DEVICE)).cpu().numpy())
    return np.concatenate(outputs)

## Question 1 — Select temporal features with Ridge

Compare recent frames, recent + daily period, and recent + daily + weekly period. Tune Ridge $lpha\in\{0.1,1,10\}$ on validation MAE. Print the explicit train/validation/test shapes for the frozen feature structure.

In [ ]:
# TODO: prepare the three lag sets and tune Ridge alpha on validation MAE.
# TODO: freeze one feature set/alpha and print all X/y split shapes.

## Question 2 — Tune a single-branch spatial CNN

Try hidden widths 16 and 32. Initialise and freeze the Ridge temporal projection, train only the spatial correction, and choose width on validation MAE. Plot the validation learning curves.

In [ ]:
# TODO: tune hidden widths 16 and 32 for the single-branch residual CNN.
# TODO: initialise from Ridge, plot learning curves and freeze the validation winner.

## Question 3 — Extension: fuse closeness, daily and weekly branches

Implement three spatial branches and fuse their feature maps:

$$\hat{Y}=\hat{Y}_{	ext{Ridge}}+f_{	ext{fuse}}([f_c(X_c),f_d(X_d),f_w(X_w)]).$$

Use the full 12-channel input and try branch widths 8 and 16. Compare the best multi-branch model with the best single branch on validation data.

In [ ]:
# TODO: implement and tune the three-branch fusion model from the formula.
# TODO: compare branch widths 8 and 16 on validation MAE.

## Question 4 — Final test comparison and error diagnosis

Evaluate each frozen family once: latest frame, Linear, Ridge, single-branch CNN and multi-branch CNN. Then plot (i) observed/predicted/residual maps for a busy frame, (ii) mean test MAE by half-hour, and (iii) spatial mean MAE. Identify where and when errors are largest and propose one concrete additional feature.

In [ ]:
# TODO: evaluate the five frozen models on test data exactly once.
# TODO: draw a busy-frame comparison, temporal MAE and spatial MAE map.
# TODO: identify one error pattern and propose a feature that could address it.